# 11: ReportLab PDF Features Testing

This notebook tests comprehensive PDF generation features:

1. **Multi-page PDFs** - Multiple sections and page breaks
2. **Table of Contents** - Auto-generated TOC
3. **Charts in Reports** - Embedded matplotlib charts
4. **Styled Tables** - Tables with branding colors
5. **Headers/Footers** - Page numbers and branding

## Prerequisites

```bash
pip install reportlab pillow matplotlib
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import tempfile

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

su.log_info("=== System Check ===")
su.log_info(f"Python: {sys.version}")

In [ ]:
# --- Path configuration ---
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")

In [ ]:
# Check ReportLab availability
try:
    from reportlab.platypus import SimpleDocTemplate, Paragraph
    from reportlab.lib.pagesizes import letter
    su.log_info("ReportLab available")
except ImportError as e:
    su.log_error(f"ReportLab NOT available: {e}")

try:
    from PIL import Image
    su.log_info("PIL (Pillow) available")
except ImportError as e:
    su.log_error(f"PIL NOT available: {e}")

try:
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend
    import matplotlib.pyplot as plt
    su.log_info(f"Matplotlib available (backend: {matplotlib.get_backend()})")
except ImportError as e:
    su.log_error(f"Matplotlib NOT available: {e}")

## 1. Basic PDF Generation

Test that basic PDF generation works.

In [ ]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Create output directory
output_dir = Path.home() / "Downloads" / "siege_reports"
output_dir.mkdir(parents=True, exist_ok=True)

# Initialize ReportGenerator
rg = ReportGenerator(
    client_name="TestClient",
    output_dir=output_dir
)

su.log_info(f"ReportGenerator initialized")
su.log_info(f"Output directory: {output_dir}")

In [ ]:
# Create sample data
np.random.seed(42)

# Quarterly sales data
sales_data = pd.DataFrame({
    'Quarter': ['Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025'],
    'Revenue ($M)': [12.5, 15.2, 18.7, 22.1],
    'Growth (%)': [8.5, 12.1, 23.0, 18.2],
    'New Customers': [145, 189, 234, 278]
})

# Regional breakdown
regional_data = pd.DataFrame({
    'Region': ['Northeast', 'Southeast', 'Midwest', 'West', 'International'],
    'Revenue ($M)': [18.2, 12.5, 8.7, 22.4, 6.7],
    'Market Share (%)': [27, 18, 13, 33, 10]
})

su.log_info("=== Sample Data Created ===")
su.log_info("Sales Data:")
su.log_info(sales_data.to_string(index=False))
su.log_info("Regional Data:")
su.log_info(regional_data.to_string(index=False))

In [ ]:
# Build basic report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Quarterly Business Report',
        'subtitle': 'FY 2025 Performance Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'TestClient'
    }
}

# Add executive summary
exec_summary = """
This report presents the quarterly performance analysis for FY 2025.
Key highlights include:

- Total revenue of $68.5M, up 15.4% year-over-year
- Strong growth in Q3 with 23% quarter-over-quarter increase
- 846 new customers acquired across all regions
- West region leads with 33% market share

The company is well-positioned for continued growth in FY 2026.
"""

report_content = rg.add_text_section(
    report_content,
    'Executive Summary',
    exec_summary
)

su.log_info("Added executive summary")

In [ ]:
# Add sales data table
report_content = rg.add_table_section(
    report_content,
    'Quarterly Performance',
    sales_data
)

# Add regional breakdown table
report_content = rg.add_table_section(
    report_content,
    'Regional Breakdown',
    regional_data
)

su.log_info(f"Added {len(report_content['sections'])} sections")

In [ ]:
# Generate basic PDF
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
basic_pdf_path = output_dir / f"basic_report_{timestamp}.pdf"

su.log_info(f"Generating: {basic_pdf_path}")

try:
    rg.generate_pdf_report(report_content, str(basic_pdf_path))
    
    if basic_pdf_path.exists():
        size_kb = basic_pdf_path.stat().st_size / 1024
        su.log_info("PDF Generated Successfully!")
        su.log_info(f"   Path: {basic_pdf_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
    else:
        su.log_warning("PDF was not created")
except Exception as e:
    su.log_error(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 2. Multi-Page PDF with Page Breaks

Test multi-page PDF generation with explicit page breaks.

In [ ]:
# Create multi-page report
multipage_content = {
    'sections': [],
    'metadata': {
        'title': 'Comprehensive Annual Report',
        'subtitle': 'Multi-Section Analysis',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
    }
}

# Section 1: Introduction (with page break after)
intro_text = """
This comprehensive report covers multiple aspects of business performance.
The analysis spans financial metrics, market position, customer analysis,
and strategic recommendations.

Report Structure:
1. Financial Performance
2. Market Analysis
3. Customer Insights
4. Strategic Recommendations
"""
multipage_content = rg.add_section(
    multipage_content,
    'text',
    'Introduction',
    intro_text,
    page_break_after=True
)

# Section 2: Financial Performance
financial_text = """
The financial performance in FY 2025 exceeded expectations across all key metrics.
Revenue grew by 15.4% compared to FY 2024, driven primarily by expansion
in the West region and successful product launches.

Key Financial Highlights:
- Total Revenue: $68.5M
- Gross Margin: 72.3%
- Operating Income: $18.2M
- Free Cash Flow: $12.8M

The company maintained strong profitability while investing in growth initiatives.
"""
multipage_content = rg.add_text_section(multipage_content, 'Financial Performance', financial_text)
multipage_content = rg.add_table_section(multipage_content, 'Quarterly Financials', sales_data)

# Add page break
multipage_content['sections'][-1]['page_break_after'] = True

# Section 3: Market Analysis
market_text = """
Market analysis reveals strong competitive positioning across all regions.
The company has successfully expanded market share in key territories
while maintaining premium pricing power.

Competitive Strengths:
- Technology leadership
- Brand recognition
- Customer satisfaction scores above industry average
- Diversified revenue streams
"""
multipage_content = rg.add_text_section(multipage_content, 'Market Analysis', market_text)
multipage_content = rg.add_table_section(multipage_content, 'Regional Market Share', regional_data)

# Section 4: Recommendations
recommendations_text = """
Based on the analysis, we recommend the following strategic priorities:

1. Continue investment in West region growth initiatives
2. Expand international presence, targeting European markets
3. Launch customer retention program to reduce churn
4. Increase R&D spending to maintain technology leadership
5. Explore strategic acquisitions in adjacent markets

These initiatives are expected to drive 20%+ revenue growth in FY 2026.
"""
multipage_content = rg.add_text_section(multipage_content, 'Strategic Recommendations', recommendations_text)

su.log_info(f"Created report with {len(multipage_content['sections'])} sections")

In [ ]:
# Generate multi-page PDF
multipage_pdf_path = output_dir / f"multipage_report_{timestamp}.pdf"

su.log_info(f"Generating: {multipage_pdf_path}")

try:
    rg.generate_pdf_report(multipage_content, str(multipage_pdf_path))
    
    if multipage_pdf_path.exists():
        size_kb = multipage_pdf_path.stat().st_size / 1024
        su.log_info("Multi-page PDF Generated!")
        su.log_info(f"   Path: {multipage_pdf_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
        su.log_info(f"   Sections: {len(multipage_content['sections'])}")
    else:
        su.log_warning("PDF was not created")
except Exception as e:
    su.log_error(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 3. Table of Contents Template

Test the dedicated Table of Contents template.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from siege_utilities.reporting.templates.table_of_contents_template import (
    TableOfContentsTemplate,
    create_table_of_contents
)

# Define TOC sections
toc_sections = [
    {
        'title': 'Executive Summary',
        'page': 3,
        'subsections': [
            {'title': 'Key Findings', 'page': 3},
            {'title': 'Recommendations', 'page': 4}
        ]
    },
    {
        'title': 'Financial Performance',
        'page': 5,
        'subsections': [
            {'title': 'Revenue Analysis', 'page': 5},
            {'title': 'Cost Structure', 'page': 6},
            {'title': 'Profitability Metrics', 'page': 7}
        ]
    },
    {
        'title': 'Market Analysis',
        'page': 8,
        'subsections': [
            {'title': 'Competitive Landscape', 'page': 8},
            {'title': 'Market Share Trends', 'page': 9}
        ]
    },
    {
        'title': 'Strategic Recommendations',
        'page': 10
    },
    {
        'title': 'Appendices',
        'page': 11
    }
]

su.log_info(f"Defined {len(toc_sections)} main sections with subsections")

In [ ]:
# Generate TOC PDF
toc_pdf_path = output_dir / f"table_of_contents_{timestamp}.pdf"

su.log_info(f"Generating: {toc_pdf_path}")

try:
    c = canvas.Canvas(str(toc_pdf_path), pagesize=letter)
    
    create_table_of_contents(
        canvas_obj=c,
        sections=toc_sections,
        title="Table of Contents",
        page_number=2
    )
    
    c.save()
    
    if toc_pdf_path.exists():
        size_kb = toc_pdf_path.stat().st_size / 1024
        su.log_info("Table of Contents PDF Generated!")
        su.log_info(f"   Path: {toc_pdf_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
    else:
        su.log_warning("PDF was not created")
except Exception as e:
    su.log_error(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 4. Charts in Reports

Test embedding matplotlib charts in PDF reports.

In [ ]:
import matplotlib.pyplot as plt

# Create sample charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: Revenue by quarter
quarters = sales_data['Quarter']
revenue = sales_data['Revenue ($M)']
colors = ['#2E5B8C', '#4A90E2', '#7FB3E8', '#B3D4F5']

axes[0].bar(quarters, revenue, color=colors)
axes[0].set_title('Quarterly Revenue', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Revenue ($M)')
axes[0].set_xlabel('Quarter')
for i, v in enumerate(revenue):
    axes[0].text(i, v + 0.5, f'${v}M', ha='center', fontsize=10)

# Pie chart: Regional market share
regions = regional_data['Region']
market_share = regional_data['Market Share (%)']
pie_colors = ['#2E5B8C', '#4A90E2', '#E74C3C', '#27AE60', '#F39C12']

axes[1].pie(market_share, labels=regions, autopct='%1.0f%%', colors=pie_colors, startangle=90)
axes[1].set_title('Regional Market Share', fontsize=14, fontweight='bold')

plt.tight_layout()

# Save chart
chart_path = output_dir / f"charts_{timestamp}.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()

su.log_info(f"Chart saved: {chart_path}")
su.log_info(f"Size: {chart_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Create report with embedded chart
from siege_utilities.reporting.templates.base_template import BaseReportTemplate
from reportlab.platypus import Paragraph, Spacer, PageBreak
from reportlab.platypus import Image as RLImage
from reportlab.lib.units import inch

chart_report_path = output_dir / f"report_with_charts_{timestamp}.pdf"

su.log_info(f"Generating: {chart_report_path}")

try:
    # Create template
    template = BaseReportTemplate(str(chart_report_path))
    
    # Build story
    story = []
    
    # Title
    story.extend(template.add_title_page(
        title="Business Analytics Report",
        subtitle="With Embedded Charts",
        author="Siege Analytics",
        date=datetime.now().strftime('%B %d, %Y')
    ))
    
    # Introduction
    story.append(Paragraph("Data Visualizations", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "The following charts illustrate key business metrics including quarterly "
        "revenue trends and regional market share distribution.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    # Embed chart
    if chart_path.exists():
        img = RLImage(str(chart_path), width=7*inch, height=3.5*inch)
        story.append(img)
        story.append(Spacer(1, 12))
        story.append(Paragraph(
            "Figure 1: Quarterly Revenue and Regional Market Share",
            template.styles['Caption'] if 'Caption' in template.styles else template.styles['Normal']
        ))
    
    # Build document
    template.build_document(story)
    
    if chart_report_path.exists():
        size_kb = chart_report_path.stat().st_size / 1024
        su.log_info("Report with Charts Generated!")
        su.log_info(f"   Path: {chart_report_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
    else:
        su.log_warning("PDF was not created")
except Exception as e:
    su.log_error(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 5. Comprehensive Report with All Features

Test a full report combining all features.

In [ ]:
from reportlab.platypus import Table, TableStyle
from reportlab.lib import colors

comprehensive_report_path = output_dir / f"comprehensive_report_{timestamp}.pdf"

su.log_info(f"Generating comprehensive report: {comprehensive_report_path}")

try:
    # Create template
    template = BaseReportTemplate(str(comprehensive_report_path))
    
    # Build story
    story = []
    
    # Title page
    story.extend(template.add_title_page(
        title="Annual Business Report",
        subtitle="FY 2025 Comprehensive Analysis",
        author="Siege Analytics",
        date=datetime.now().strftime('%B %d, %Y')
    ))
    
    # Executive Summary
    story.append(Paragraph("Executive Summary", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "This comprehensive report presents the financial and operational performance "
        "for FY 2025. The company achieved strong results across all key metrics, "
        "with revenue growing 15.4% year-over-year to $68.5M.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    # Quarterly Performance Table
    story.append(Paragraph("Quarterly Performance", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    
    # Convert DataFrame to table data
    table_data = [sales_data.columns.tolist()] + sales_data.values.tolist()
    
    # Create styled table
    t = Table(table_data)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2E5B8C')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('TOPPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#F5F5F5')),
        ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CCCCCC')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 8),
        ('TOPPADDING', (0, 1), (-1, -1), 8),
    ]))
    story.append(t)
    story.append(Spacer(1, 24))
    
    # Add chart
    story.append(Paragraph("Data Visualizations", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    if chart_path.exists():
        img = RLImage(str(chart_path), width=6*inch, height=3*inch)
        story.append(img)
    story.append(Spacer(1, 12))
    
    # Page break
    story.append(PageBreak())
    
    # Regional Analysis
    story.append(Paragraph("Regional Analysis", template.styles['Heading1']))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        "The regional breakdown shows strong performance across all territories. "
        "The West region continues to lead with 33% market share.",
        template.styles['Normal']
    ))
    story.append(Spacer(1, 24))
    
    # Regional table
    regional_table_data = [regional_data.columns.tolist()] + regional_data.values.tolist()
    t2 = Table(regional_table_data)
    t2.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#27AE60')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
        ('TOPPADDING', (0, 0), (-1, 0), 12),
        ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#E8F5E9')),
        ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CCCCCC')),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 8),
        ('TOPPADDING', (0, 1), (-1, -1), 8),
    ]))
    story.append(t2)
    story.append(Spacer(1, 24))
    
    # Recommendations
    story.append(Paragraph("Strategic Recommendations", template.styles['Heading2']))
    story.append(Spacer(1, 12))
    recommendations = [
        "1. Continue investment in West region growth initiatives",
        "2. Expand international presence targeting European markets",
        "3. Launch customer retention program to reduce churn",
        "4. Increase R&D spending to maintain technology leadership",
        "5. Explore strategic acquisitions in adjacent markets"
    ]
    for rec in recommendations:
        story.append(Paragraph(rec, template.styles['Normal']))
        story.append(Spacer(1, 6))
    
    # Build document
    template.build_document(story)
    
    if comprehensive_report_path.exists():
        size_kb = comprehensive_report_path.stat().st_size / 1024
        su.log_info("Comprehensive Report Generated!")
        su.log_info(f"   Path: {comprehensive_report_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
    else:
        su.log_warning("PDF was not created")
except Exception as e:
    su.log_error(f"Error: {e}")
    import traceback
    traceback.print_exc()

## Summary

List all generated PDFs.

In [ ]:
su.log_info("=" * 60)
su.log_info("REPORTLAB PDF FEATURE TEST RESULTS")
su.log_info("=" * 60)

# Check all generated files
generated_files = [
    ('Basic PDF', basic_pdf_path if 'basic_pdf_path' in dir() else None),
    ('Multi-page PDF', multipage_pdf_path if 'multipage_pdf_path' in dir() else None),
    ('Table of Contents', toc_pdf_path if 'toc_pdf_path' in dir() else None),
    ('Charts Report', chart_report_path if 'chart_report_path' in dir() else None),
    ('Comprehensive Report', comprehensive_report_path if 'comprehensive_report_path' in dir() else None),
]

passed = 0
for name, path in generated_files:
    if path and path.exists():
        size_kb = path.stat().st_size / 1024
        su.log_info(f"  {name}: {size_kb:.1f} KB")
        passed += 1
    else:
        su.log_warning(f"  {name}: Not generated")

su.log_info(f"{passed}/{len(generated_files)} PDFs generated successfully")

if passed == len(generated_files):
    su.log_info("All ReportLab PDF features working!")
else:
    su.log_warning("Some features need attention")

su.log_info(f"Output directory: {output_dir}")